# UCI HAR — Transition-Centred Data Split

## Why transition-centred?

| | Independent window classification (incorrect) | Transition-centred (this notebook) |
|---|---|---|
| Label | Which activity is happening now? | Which activity are we **entering**? |
| Window placement | Every overlapping window equally | Windows at activity **change points** |
| None class | Not used | Windows in stable activity regions |
| Analogy | N/A | North Sea: window centred on formation top |

This mirrors exactly how the North Sea pipeline works:
- **Subject** = wellbore
- **Activity transition** = formation top (geological boundary)
- **New activity class** = geological formation being entered
- **None class** = windows in stable regions away from any transition

The model therefore learns: *at this sensor window, is an activity change occurring — and if so, which new activity is beginning?*

**Input:** `../UCI HAR Dataset/` (raw inertial signal `.txt` files)

**Output** (written to `har_transition/`):
| File | Content |
|---|---|
| `train.npz` | `X` (N, 9, 128) float32, `y` (N,) int64 — 0=None, 1–6=new activity |
| `val.npz` | same format |
| `test.npz` | same format |
| `warm_state.json` | metadata consumed by `experiment_training_medical.ipynb` |

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
import os

# looks for the data in $UCI_HAR_DIR first, then <repo>/datasets/medical/UCI HAR Dataset,
# then a few paths relative to where the notebook is run (see datasets/medical/README.md)

def _find_repo_root():
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (parent / "setup.py").is_file() and (parent / "geoxts").is_dir():
            return parent
    return None

_REPO_ROOT = _find_repo_root()
_HAR_CANDIDATES = []
if os.environ.get("UCI_HAR_DIR"):
    _HAR_CANDIDATES.append(Path(os.environ["UCI_HAR_DIR"]))
if _REPO_ROOT is not None:
    _HAR_CANDIDATES.append(_REPO_ROOT / "datasets" / "medical" / "UCI HAR Dataset")
_HAR_CANDIDATES += [
    Path("../UCI HAR Dataset"),                # from datasets/medical/Classification/
    Path("UCI HAR Dataset"),                   # from datasets/medical/ or the VM home dir
    Path("datasets/medical/UCI HAR Dataset"),  # from the repo root
]
HAR_ROOT = next((p for p in _HAR_CANDIDATES if p.exists()), _HAR_CANDIDATES[0])
if not HAR_ROOT.exists():
    raise FileNotFoundError(
        f"UCI HAR Dataset not found at:\n  {HAR_ROOT.resolve()}\nUpdate HAR_ROOT."
    )

SAVE_DIR = Path("har_transition")
SAVE_DIR.mkdir(exist_ok=True)

# 0 = None (no transition), 1-6 = the activity being entered.
# Same layout as North Sea, where it's ['None'] + formation names
NONE_LABEL = 0
ACTIVITY_LABELS = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING",
]
PRED_COLUMN = ["None"] + ACTIVITY_LABELS
N_CLASSES = len(PRED_COLUMN)  # 7

# 9 inertial signals, 128 timesteps per window
CHANNELS = [
    "body_acc_x",  "body_acc_y",  "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]
N_CHANNELS = len(CHANNELS)
WINDOW_LEN = 128

# test subjects (2,4,9,10,12,13,18,20,24) are fixed by the dataset authors.
# 4 of the 21 train subjects are held out for validation, about 20% of train windows
VAL_SUBJECTS = {22, 25, 27, 29}

# windows this close to a transition are ambiguous, so they're left out of the None pool
TRANSITION_MARGIN = 2
NONE_RATIO = 1.5  # None windows per transition window

RANDOM_SEED = 42

print(f"Classes ({N_CLASSES}): {PRED_COLUMN}")
print(f"Channels ({N_CHANNELS}): {CHANNELS}")
print(f"Save dir: {SAVE_DIR.resolve()}")

In [3]:
def load_raw_split(split: str):
    """Load the 9 raw signal channels for 'train' or 'test', in recording order.

    Returns X (N, 9, 128), y (N,) with labels shifted from 1-6 to 0-5, and subject IDs (N,).
    """
    split_dir = HAR_ROOT / split
    sig_dir   = split_dir / "Inertial Signals"

    arrays = []
    for ch in CHANNELS:
        arr = np.loadtxt(sig_dir / f"{ch}_{split}.txt", dtype=np.float32)  # (N, 128)
        arrays.append(arr)
    X = np.stack(arrays, axis=1)  # (N, 9, 128)

    y_raw    = np.loadtxt(split_dir / f"y_{split}.txt",       dtype=np.int64)
    y        = y_raw - 1  # 1-6 -> 0-5
    subjects = np.loadtxt(split_dir / f"subject_{split}.txt", dtype=np.int64)

    return X, y, subjects


print("Loading train split...")
X_tr_raw, y_tr_raw, subj_tr = load_raw_split("train")
print(f"  Raw train: X={X_tr_raw.shape}, y={y_tr_raw.shape}")

print("Loading test split...")
X_te_raw, y_te_raw, subj_te = load_raw_split("test")
print(f"  Test:      X={X_te_raw.shape}, y={y_te_raw.shape}")

Loading train split...
  Raw train: X=(7352, 9, 128), y=(7352,)
Loading test split...
  Test:      X=(2947, 9, 128), y=(2947,)


In [4]:
# split by subject so no person appears in more than one set
val_mask   = np.isin(subj_tr, sorted(VAL_SUBJECTS))
train_mask = ~val_mask

X_train, y_train, subj_train = X_tr_raw[train_mask], y_tr_raw[train_mask], subj_tr[train_mask]
X_val,   y_val,   subj_val   = X_tr_raw[val_mask],   y_tr_raw[val_mask],   subj_tr[val_mask]
X_test,  y_test,  subj_test  = X_te_raw, y_te_raw, subj_te

print(f"Train: {X_train.shape}  subjects={sorted(np.unique(subj_train).tolist())}")
print(f"Val:   {X_val.shape}  subjects={sorted(np.unique(subj_val).tolist())}")
print(f"Test:  {X_test.shape}  subjects={sorted(np.unique(subj_test).tolist())}")

Train: (5902, 9, 128)  subjects=[1, 3, 5, 6, 7, 8, 11, 14, 15, 16, 17, 19, 21, 23, 26, 28, 30]
Val:   (1450, 9, 128)  subjects=[22, 25, 27, 29]
Test:  (2947, 9, 128)  subjects=[2, 4, 9, 10, 12, 13, 18, 20, 24]


In [5]:
# per-channel z-score, using train statistics only
ch_mean = X_train.mean(axis=(0, 2), keepdims=True).astype(np.float32)
ch_std  = X_train.std( axis=(0, 2), keepdims=True).clip(min=1e-8).astype(np.float32)

X_train = (X_train - ch_mean) / ch_std
X_val   = (X_val   - ch_mean) / ch_std
X_test  = (X_test  - ch_mean) / ch_std

print("Normalisation applied (fit on train, applied to val + test).")
for i, ch in enumerate(CHANNELS):
    print(f"  {ch:<14}  mean={X_train[:, i, :].mean():+.4f}  std={X_train[:, i, :].std():.4f}")

Normalisation applied (fit on train, applied to val + test).
  body_acc_x      mean=+0.0000  std=1.0000
  body_acc_y      mean=-0.0000  std=1.0000
  body_acc_z      mean=-0.0000  std=1.0000
  body_gyro_x     mean=+0.0000  std=1.0000
  body_gyro_y     mean=-0.0000  std=1.0000
  body_gyro_z     mean=-0.0000  std=1.0000
  total_acc_x     mean=+0.0000  std=1.0000
  total_acc_y     mean=+0.0000  std=1.0000
  total_acc_z     mean=-0.0000  std=1.0000


In [6]:
def extract_transition_dataset(X, y, subjects, none_ratio=NONE_RATIO,
                               margin=TRANSITION_MARGIN, seed=RANDOM_SEED):
    """HAR version of geoxts extract_dataset_Xy.

    For each subject, the first window of every new activity is a transition window
    (label 1-6). Windows more than `margin` steps from any transition go into the None
    pool (label 0), which is subsampled to `none_ratio` x the number of transitions.
    Returns shuffled X (M, 9, 128) and y (M,).
    """
    X_trans, y_trans = [], []
    X_stable, _      = [], []

    for subj in np.unique(subjects):
        mask = subjects == subj
        X_s  = X[mask]   # still in recording order
        y_s  = y[mask]
        N_s  = len(X_s)

        # indices where the label changes
        trans_set = set((np.where(np.diff(y_s) != 0)[0] + 1).tolist())

        for t in sorted(trans_set):
            X_trans.append(X_s[t])
            y_trans.append(int(y_s[t]) + 1)  # 0-5 -> 1-6, 0 is kept for None

        for i in range(N_s):
            if not any(abs(i - t) <= margin for t in trans_set):
                X_stable.append(X_s[i])

    X_trans  = np.array(X_trans,  dtype=np.float32)
    y_trans  = np.array(y_trans,  dtype=np.int64)
    X_stable = np.array(X_stable, dtype=np.float32)

    # subsample None so it doesn't swamp the transitions
    n_none = min(len(X_stable), int(len(X_trans) * none_ratio))
    rng    = np.random.default_rng(seed)
    idx    = rng.choice(len(X_stable), n_none, replace=False)
    X_none = X_stable[idx]
    y_none = np.zeros(n_none, dtype=np.int64)

    X_out = np.concatenate([X_trans, X_none], axis=0)
    y_out = np.concatenate([y_trans, y_none], axis=0)
    perm  = rng.permutation(len(X_out))

    return X_out[perm], y_out[perm]


print("Building transition datasets...")
X_tr_out, y_tr_out = extract_transition_dataset(X_train, y_train, subj_train)
X_vl_out, y_vl_out = extract_transition_dataset(X_val,   y_val,   subj_val)
X_te_out, y_te_out = extract_transition_dataset(X_test,  y_test,  subj_test)

print(f"Train: X={X_tr_out.shape}  y={y_tr_out.shape}")
print(f"Val:   X={X_vl_out.shape}  y={y_vl_out.shape}")
print(f"Test:  X={X_te_out.shape}  y={y_te_out.shape}")

Building transition datasets...
Train: X=(532, 9, 128)  y=(532,)
Val:   X=(115, 9, 128)  y=(115,)
Test:  X=(277, 9, 128)  y=(277,)


In [7]:
for split_name, y_s in [("Train", y_tr_out), ("Val", y_vl_out), ("Test", y_te_out)]:
    classes, counts = np.unique(y_s, return_counts=True)
    total = len(y_s)
    print(f"\n{split_name} ({total} windows):")
    for c, n in zip(classes, counts):
        label = PRED_COLUMN[c]
        print(f"  [{c}] {label:<22}  {n:>5}  ({100*n/total:.1f}%)")


Train (532 windows):
  [0] None                      319  (60.0%)
  [1] WALKING                    34  (6.4%)
  [2] WALKING_UPSTAIRS           45  (8.5%)
  [3] WALKING_DOWNSTAIRS         47  (8.8%)
  [4] SITTING                    35  (6.6%)
  [5] STANDING                   17  (3.2%)
  [6] LAYING                     35  (6.6%)

Val (115 windows):
  [0] None                       69  (60.0%)
  [1] WALKING                     8  (7.0%)
  [2] WALKING_UPSTAIRS            9  (7.8%)
  [3] WALKING_DOWNSTAIRS          9  (7.8%)
  [4] SITTING                     8  (7.0%)
  [5] STANDING                    4  (3.5%)
  [6] LAYING                      8  (7.0%)

Test (277 windows):
  [0] None                      166  (59.9%)
  [1] WALKING                    18  (6.5%)
  [2] WALKING_UPSTAIRS           23  (8.3%)
  [3] WALKING_DOWNSTAIRS         25  (9.0%)
  [4] SITTING                    18  (6.5%)
  [5] STANDING                    9  (3.2%)
  [6] LAYING                     18  (6.5%)


In [8]:
np.savez_compressed(SAVE_DIR / "train.npz", X=X_tr_out, y=y_tr_out)
np.savez_compressed(SAVE_DIR / "val.npz",   X=X_vl_out, y=y_vl_out)
np.savez_compressed(SAVE_DIR / "test.npz",  X=X_te_out, y=y_te_out)

# same keys as the North Sea warm_state, so experiment_training_medical can load it
# the same way. Subjects take the place of wells
train_subjects = sorted(np.unique(subj_train).tolist())
val_subjects   = sorted(np.unique(subj_val).tolist())
test_subjects  = sorted(np.unique(subj_test).tolist())
all_subjects   = sorted(set(train_subjects) | set(val_subjects) | set(test_subjects))
subject2id     = {str(s): i for i, s in enumerate(all_subjects)}

warm_state = {
    "MARKERS":       ACTIVITY_LABELS,
    "PRED_COLUMN":   PRED_COLUMN,
    "FEATURES_GEOX": CHANNELS,
    "WELL_COL":      "subject_id",
    "N_CLASSES":     N_CLASSES,
    "WINDOW_LEN":    WINDOW_LEN,
    "train_wells":   train_subjects,
    "val_wells":     val_subjects,
    "test_wells":    test_subjects,
    "well2id":       subject2id,
    "ch_mean":  ch_mean.flatten().tolist(),
    "ch_std":   ch_std.flatten().tolist(),
    "none_ratio":         NONE_RATIO,
    "transition_margin":  TRANSITION_MARGIN,
    # depth flags are North Sea only, set so the shared loader skips them
    "USE_DEPTH":     False,
    "DEPTH_BOUNDS":  None,
    "dataset_type":  "medical_transition",
}

with open(SAVE_DIR / "warm_state.json", "w") as f:
    json.dump(warm_state, f, indent=2)

print(f"Saved to {SAVE_DIR}/")
print(f"  train.npz  X={X_tr_out.shape}  y={y_tr_out.shape}")
print(f"  val.npz    X={X_vl_out.shape}  y={y_vl_out.shape}")
print(f"  test.npz   X={X_te_out.shape}  y={y_te_out.shape}")
print(f"  warm_state.json")

Saved to har_transition/
  train.npz  X=(532, 9, 128)  y=(532,)
  val.npz    X=(115, 9, 128)  y=(115,)
  test.npz   X=(277, 9, 128)  y=(277,)
  warm_state.json


In [9]:
# reload the saved files and check shapes and labels
for split_name, fname in [("train", "train.npz"), ("val", "val.npz"), ("test", "test.npz")]:
    d = np.load(SAVE_DIR / fname)
    X, y = d['X'], d['y']
    assert X.ndim == 3 and X.shape[1] == 9 and X.shape[2] == 128, f"Unexpected shape: {X.shape}"
    assert y.min() >= 0 and y.max() <= 6, f"Label out of range: {y.min()}-{y.max()}"
    classes, counts = np.unique(y, return_counts=True)
    print(f"{split_name:<6}  X={X.shape}  y={y.shape}  classes={classes.tolist()}  counts={counts.tolist()}")

with open(SAVE_DIR / "warm_state.json") as f:
    ws = json.load(f)
assert ws["dataset_type"] == "medical_transition"
assert len(ws["FEATURES_GEOX"]) == 9
assert len(ws["PRED_COLUMN"]) == 7
print("\nAll checks passed.")

train   X=(532, 9, 128)  y=(532,)  classes=[0, 1, 2, 3, 4, 5, 6]  counts=[319, 34, 45, 47, 35, 17, 35]
val     X=(115, 9, 128)  y=(115,)  classes=[0, 1, 2, 3, 4, 5, 6]  counts=[69, 8, 9, 9, 8, 4, 8]
test    X=(277, 9, 128)  y=(277,)  classes=[0, 1, 2, 3, 4, 5, 6]  counts=[166, 18, 23, 25, 18, 9, 18]

All checks passed.
